### 0. Reflection 的任務設計

#### 🌟 任務說明：程式優化小幫手

**🎯 流程說明：**
1. 使用者輸入今天想寫的程式
2. `model_writer` 生成第一版程式碼
3. `model_reviewer` 檢查內容是否達到**Correctness、Readability、Efficiency、Edge Cases、Best Practices**的標準
4. `model_writer` 根據建議產出第二版
5. Gradio 呈現：三個欄位：第一版、建議、第二版

#### 1. 讀入你的金鑰


In [ ]:
import os
from google.colab import userdata

In [ ]:
#【使用 Groq】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
provider = "groq"
model = "openai/gpt-oss-120b"

In [ ]:
!pip install aisuite[all]

### 2. 基本的設定

In [ ]:
import aisuite as ai

In [ ]:
provider_writer = "groq"
model_writer="openai/gpt-oss-120b"

provider_reviewer = "groq"
model_reviewer = "openai/gpt-oss-120b"

標準回應函式

In [ ]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider="groq",
          model="openai/gpt-oss-120b"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)
    return response.choices[0].message.content

####  3. 設定「作者」和「審查員」

In [ ]:
system_writer = """
You are a Python Code Generator.
Based *only* on the user's request, write Python code that fulfills the requirement.
Output *only* the complete Python code, *NOT enclosed in triple backticks (```python ... ```)*.
Do not add any other text before or after the code block.
"""
system_reviewer = """
You are an expert Python Code Reviewer.
Your task is to provide constructive feedback on the provided code.

    **Code to Review:**
    ```python
    {first_version}
    ```

**Review Criteria:**
1.  **Correctness:** Does the code work as intended? Are there logic errors?
2.  **Readability:** Is the code clear and easy to understand? Follows PEP 8 style guidelines?
3.  **Efficiency:** Is the code reasonably efficient? Any obvious performance bottlenecks?
4.  **Edge Cases:** Does the code handle potential edge cases or invalid inputs gracefully?
5.  **Best Practices:** Does the code follow common Python best practices?

**Output:**
Provide your feedback as a concise, bulleted list. Focus on the most important points for improvement.
If the code is excellent and requires no changes, simply state: "No major issues found."
Output *only* the review comments or the "No major issues" statement.
"""

In [ ]:
def reflect_post(prompt):
    # Step 1: Writer 初稿
    first_version = reply(system_writer, prompt,
                          provider=provider_writer,
                          model=model_writer
                          )

    # Step 2: Reviewer 給建議
    suggestion = reply(system_reviewer, first_version,
                       provider=provider_reviewer,
                       model=model_reviewer
                       )

    # Step 3: Writer 再寫一次（根據建議）
    second_prompt = f"這是我剛剛寫的程式碼：\n{first_version}\n\n這是修改建議：\n{suggestion}\n\n請根據這些建議，幫我改得更好。請用台灣習慣的中文, 並且只要輸出改好的程式碼就可以了。"
    second_version = reply(system_writer, second_prompt,
                           provider=provider_writer,
                           model=model_writer
                           )

    return first_version, suggestion, second_version

### 4. 用 Gradio 打造你的對話機器人 Web App!

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("### 🤖 程式優化小幫手（Reflection Agent）")
    user_input = gr.Textbox(label="請輸入你今天寫的程式")
    btn = gr.Button("生成程式碼 & 修正建議")

    with gr.Row():
        out1 = gr.Textbox(label="🌟 第一版程式碼 (model_writer)",lines=10)
        out2 = gr.Textbox(label="🧐 修改建議 (model_reviewer)",lines=10)
        out3 = gr.Textbox(label="✨ 第二版程式碼 (model_writer 改寫)",lines=10)

    btn.click(reflect_post, inputs=[user_input], outputs=[out1, out2, out3])

In [ ]:
demo.launch(share=True, debug=True)